# Lab 06 — Real distribution: two processes, a real wire

**Goal:** run the pipeline as **two OS processes** exchanging the boundary tensor with `torch.distributed` (gloo backend = CPU/Ethernet), add GPipe-style micro-batching, time the transfers, and (on Linux with sudo) throttle the link with `tc netem` to emulate your consumer network.

Works on Colab (2 CPU processes). The mechanics are identical on 2 physical machines — only the init method changes.

## 1. Concepts in 60 seconds
- A **process group** connects N processes; each has a **rank** (0..N-1).
- `dist.send(tensor, dst)` / `dist.recv(tensor, src)` are blocking point-to-point ops — exactly a pipeline boundary.
- **gloo** runs over TCP (your Ethernet story); **nccl** is the GPU/NVLink backend.
- `torchrun --nproc_per_node=2 file.py` launches both ranks and wires the group up.

In [ ]:
%%writefile pipe_demo.py
import torch, torch.nn as nn, torch.distributed as dist, time, os

dist.init_process_group("gloo")
rank = dist.get_rank()
torch.manual_seed(0)                    # same init on both ranks

B, D, H, C, MICRO = 64, 64, 256, 10, 4
net1 = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, H))
net2 = nn.Sequential(nn.Linear(H, H), nn.ReLU(), nn.Linear(H, C))
opt = torch.optim.AdamW((net1 if rank == 0 else net2).parameters(), lr=1e-3)

X = torch.randn(B, D); Yt = torch.randint(0, C, (B,))   # same data both ranks (seeded)

for step in range(20):
    opt.zero_grad()
    for mb in range(MICRO):                              # GPipe: all micro-batches fwd+bwd, then ONE step
        x  = X[mb*B//MICRO:(mb+1)*B//MICRO]
        yt = Yt[mb*B//MICRO:(mb+1)*B//MICRO]
        if rank == 0:
            h = net1(x)
            dist.send(h.detach(), dst=1)                 # ---- forward wire ----
            g = torch.empty_like(h)
            dist.recv(g, src=1)                          # ---- backward wire ----
            h.backward(g / MICRO)                        # accumulate (mean over micro-batches)
        else:
            h = torch.empty(x.size(0), H)
            dist.recv(h, src=0)
            h.requires_grad_()
            loss = nn.functional.cross_entropy(net2(h), yt)
            (loss / MICRO).backward()
            dist.send(h.grad * MICRO, dst=0)
            if mb == MICRO-1 and step % 5 == 0:
                print(f"step {step:2d}  loss {loss.item():.4f}", flush=True)
    opt.step()

if rank == 0:                                            # bandwidth measurement
    t = torch.randn(1_000_000)                           # 4 MB fp32
    t0 = time.perf_counter()
    for _ in range(20): dist.send(t, dst=1)
    dt = time.perf_counter() - t0
    print(f"[rank0] {20*t.numel()*4/1e6/dt:.0f} MB/s over this 'wire'", flush=True)
else:
    t = torch.empty(1_000_000)
    for _ in range(20): dist.recv(t, src=0)
dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 pipe_demo.py

Loss falls → two processes are jointly training one model with only boundary tensors crossing between them. The `MB/s` line is loopback speed — absurdly fast, because there's no real network yet.

## 2. Emulate the consumer network — `tc netem`
This is your entire 'experimental cluster' until real hardware exists: dial any bandwidth and latency on the loopback device. (Linux + sudo; works on Colab. **Always run the delete cell after.**)

In [ ]:
!sudo tc qdisc add dev lo root netem rate 1gbit delay 0.2ms 2>/dev/null || sudo tc qdisc change dev lo root netem rate 1gbit delay 0.2ms
!tc qdisc show dev lo

In [ ]:
!torchrun --nproc_per_node=2 pipe_demo.py    # rerun: watch MB/s collapse toward ~110-120

In [ ]:
!sudo tc qdisc del dev lo root    # ALWAYS restore the loopback when done
!tc qdisc show dev lo

You now command a programmable network: `rate 100mbit`, `delay 50ms` (federated-WAN emulation), even `delay 10ms 5ms` for jitter. This is the mechanism behind the proposal's heterogeneity sweep.

## Exercises
1. Move the quantizer from Lab 04 into `pipe_demo.py` and quantize both wires (fwd 4-bit, bwd 8-bit). Confirm loss still falls. You have now built the proposal's data path end-to-end.
2. Time one full step at `rate 1gbit` vs `100mbit` vs no netem. Where did the time go? (Communication wall, measured.)
3. Replace the MLPs with GPT-2 stage1/stage2 from Lab 05 (CPU, S=64) — identical structure, real model.
4. Two physical machines: replace `torchrun` single-node launch with `--nnodes=2 --node_rank={0,1} --master_addr=<ip-of-rank0>` and run over your actual LAN.